In [ ]:
!pip install -q transformers accelerate bitsandbytes peft trl
!pip install -q rasterio tifffile albumentations
!pip install -q huggingface_hub
!pip install datasets==2.19.2 -q

## CHECK BELOW FOR EMERGENCY NOTE

## Test Dataset

In [ ]:
import os
import gc
import torch
from PIL import Image
from datasets import load_dataset
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

## Environment and Authentication

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token

HF_USERNAME = "nillohitroy"
DATASET_REPO = f"{HF_USERNAME}/satquery-multitask-dataset"
OUTPUT_MODEL_REPO = f"{HF_USERNAME}/satquery-llava-multitask"
BASE_MODEL_ID = "llava-hf/llava-1.5-7b-hf"

## Load Processor and 4-bit Quantized Model

In [ ]:
print(f"\nInitializing 4-bit LLaVA Base Model: {BASE_MODEL_ID}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

processor = AutoProcessor.from_pretrained(BASE_MODEL_ID, token=hf_token)

# Ensure consistent padding token
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    attn_implementation="eager",
    token=hf_token
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# Expanded LoRA targeting attention and MLP projections
lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Stream Data from Huggingface

In [ ]:
print(f"\nFetching dataset from Hugging Face: {DATASET_REPO}...")
hf_dataset = load_dataset(DATASET_REPO, split="train", token=hf_token)
print(f"Loaded {len(hf_dataset)} total training samples.")

class MultiTaskVQADataset(torch.utils.data.Dataset):
    def __init__(self, dataset, processor):
        self.dataset = dataset
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        
        try:
            img = item['image']
            if not isinstance(img, Image.Image):
                img = Image.open(img)
            img = img.convert('RGB')
        except Exception:
            return self._get_fallback()

        question = item.get('question', 'Describe this satellite image.')
        answer = item.get('answer', 'Unknown.')
        prompt = f"USER: <image>\n{question}\nASSISTANT: {answer}"

        try:
            # max_length=512 prevents long sequences from exploding backward-pass VRAM
            inputs = self.processor(
                text=prompt,
                images=img,
                max_length=512,
                truncation=True,
                return_tensors="pt"
            )
            return {k: v.squeeze(0) for k, v in inputs.items()}
        except Exception:
            return self._get_fallback()

    def _get_fallback(self):
        # Native LLaVA-1.5 CLIP resolution
        img = Image.new('RGB', (336, 336), color='black')
        prompt = "USER: <image>\nDescribe.\nASSISTANT: Empty."
        inputs = self.processor(text=prompt, images=img, return_tensors="pt")
        return {k: v.squeeze(0) for k, v in inputs.items()}

train_dataset = MultiTaskVQADataset(hf_dataset, processor)

## Collator with Loss Masking

In [ ]:
def multimodal_collate_fn(examples):
    input_ids_list = [example["input_ids"] for example in examples]
    attention_mask_list = [example["attention_mask"] for example in examples]

    input_ids = torch.nn.utils.rnn.pad_sequence(
        input_ids_list,
        batch_first=True,
        padding_value=processor.tokenizer.pad_token_id
    )

    attention_mask = torch.nn.utils.rnn.pad_sequence(
        attention_mask_list,
        batch_first=True,
        padding_value=0
    )

    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    image_token_id = processor.tokenizer.convert_tokens_to_ids("<image>")

    labels_list = []
    for example in examples:
        label = example["input_ids"].clone()
        # Mask <image> token so loss is strictly computed on language reasoning
        if image_token_id is not None:
            label[label == image_token_id] = -100
        labels_list.append(label)

    labels = torch.nn.utils.rnn.pad_sequence(
        labels_list,
        batch_first=True,
        padding_value=-100
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "pixel_values": pixel_values,
        "labels": labels
    }

## Training

In [ ]:
CHECKPOINT_DIR = "/kaggle/working/satquery-lora-expanded"

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=100,
    max_steps=1200,             # ~9,600 samples seen; converges in ~3.5h on T4
    logging_steps=25,
    save_strategy="steps",
    save_steps=300,
    save_total_limit=2,         # Critical: keeps only 2 checkpoints to prevent disk full error
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    dataloader_num_workers=2,
    dataloader_pin_memory=False # Avoids pinned CUDA memory OOM
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=multimodal_collate_fn,
)

print("\nStarting LoRA Fine-Tuning...")
trainer.train()

# Save final adapter locally
final_adapter_dir = "/kaggle/working/final_adapter"
model.save_pretrained(final_adapter_dir)
processor.save_pretrained(final_adapter_dir)
print(f"\nFinal adapter weights saved to {final_adapter_dir}")

# Push adapter directly to Hub as immediate backup
ADAPTER_REPO = f"{OUTPUT_MODEL_REPO}-adapter"
print(f"Backing up LoRA adapter to: {ADAPTER_REPO}...")
model.push_to_hub(ADAPTER_REPO, token=hf_token)
processor.push_to_hub(ADAPTER_REPO, token=hf_token)

## GPU Based Merge

In [1]:
!pip install -q --upgrade torchao
!pip install -q transformers peft accelerate bitsandbytes huggingface_hub

## Emergency: DO NOT RUN THIS. RUN THE CELL BELOW.

In [3]:
import os
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# 2. Authenticate
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

BASE_MODEL_ID = "llava-hf/llava-1.5-7b-hf"
ADAPTER_REPO = "nillohitroy/satquery-llava-multitask-adapter"
FINAL_REPO = "nillohitroy/satquery-llava-multitask"

print("\n" + "="*50)
print("DOWNLOADING & MERGING MODEL ON CPU")
print("="*50)

print(f"1. Loading Base Model ({BASE_MODEL_ID}) into CPU RAM...")
# device_map="cpu" ensures we don't touch the limited GPU VRAM
base_model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu", 
    low_cpu_mem_usage=True,
    token=hf_token
)

print(f"2. Downloading your trained LoRA adapter from {ADAPTER_REPO}...")
model_to_merge = PeftModel.from_pretrained(
    base_model, 
    ADAPTER_REPO, 
    token=hf_token
)

print("3. Merging weights (this takes a few minutes)...")
merged_model = model_to_merge.merge_and_unload()

print("\n" + "="*50)
print("PUSHING FINAL MODEL TO HUGGING FACE")
print("="*50)

print(f"Pushing merged standalone model to {FINAL_REPO}...")
merged_model.push_to_hub(
    FINAL_REPO,
    max_shard_size="4GB",
    token=hf_token
)

# 2. Push the processor
print("Pushing processor...")
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID, token=hf_token)
processor.push_to_hub(FINAL_REPO, token=hf_token)

print(f"\nSuccess! Your final, merged model is now published at: https://huggingface.co/{FINAL_REPO}")


DOWNLOADING & MERGING MODEL ON CPU
1. Loading Base Model (llava-hf/llava-1.5-7b-hf) into CPU RAM...


Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

2. Downloading your trained LoRA adapter from nillohitroy/satquery-llava-multitask-adapter...
3. Merging weights (this takes a few minutes)...

PUSHING FINAL MODEL TO HUGGING FACE
Pushing merged standalone model to nillohitroy/satquery-llava-multitask...


Writing model shards:   0%|          | 0/4 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Pushing processor...


No files have been modified since last commit. Skipping to prevent empty commit.



Success! Your final, merged model is now published at: https://huggingface.co/nillohitroy/satquery-llava-multitask


In [ ]:
print("\n" + "="*50)
print("PURGING TRAINING VRAM FOR GPU MERGING")
print("="*50)

del model, trainer, train_dataset, hf_dataset
gc.collect()
torch.cuda.empty_cache()

print(f"VRAM remaining after cleanup: {torch.cuda.memory_allocated() / (1024**3):.2f} GB")

print("Loading 16-bit Base Model onto GPU...")
base_model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cuda:0",
    low_cpu_mem_usage=True,
    token=hf_token
)

print("Merging LoRA weights into Base Model on GPU...")
model_to_merge = PeftModel.from_pretrained(base_model, final_adapter_dir)
merged_model = model_to_merge.merge_and_unload()

print(f"\nPushing full merged model to Hugging Face: {OUTPUT_MODEL_REPO}...")
merged_model.push_to_hub(
    OUTPUT_MODEL_REPO,
    max_shard_size="4GB",
    safe_serialization=True,
    token=hf_token
)
processor.push_to_hub(OUTPUT_MODEL_REPO, token=hf_token)

print(f"\nSuccess!!")